In [2]:
## IMPORT REQUIRED LIBRARIES

import pandas as pd
import duckdb

print("Libraries Loaded")

Libraries Loaded


In [3]:
## LOAD THE DATASET

df = pd.read_csv("../data/processed/hotel_bookings_cleaned.csv")

print(df.shape)

df.head()

(119390, 36)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_nights,total_guests,arrival_date_num,stay_type
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,0,2.0,7,Short Stay
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,0,2.0,7,Short Stay
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,1,1.0,7,Short Stay
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,1,1.0,7,Short Stay
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,2,2.0,7,Short Stay


In [4]:
## REGISTER DATABASE INTO DUCKDB

duckdb.register(
    "hotel_data",
    df
)

print("Table Registered")

Table Registered


In [5]:
## BOOKING COUNT BY HOTEL type

duckdb.sql("""      
SELECT 
    hotel, 
    COUNT(*) AS total_bookings
FROM hotel_data
GROUP BY hotel
ORDER BY total_bookings DESC          
""").df()


,hotel,total_bookings
0,City Hotel,79330
1,Resort Hotel,40060


### Observation
- City Hotels account for approx. 66.45% of total bookings.
- Resort Hotels account for approx. 33.55% of total bookings.
- City Hotels receive nearly twice as many bookings as Resort Hotels.
- This suggests that urban destinations generate substantially higher booking demand than resort destinations.

In [6]:
## CANCELLATION RATE BY HOTEL

duckdb.sql("""
SELECT 
    hotel, 
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY hotel
ORDER BY cancellation_rate DESC
""").df()


,hotel,total_bookings,cancelled_bookings,cancellation_rate
0,City Hotel,79330,33102.0,41.73
1,Resort Hotel,40060,11122.0,27.76


### Observation
- City Hotels experience substantially higher cancellation rate of 41.73%. 
- Resort Hotels show a significantly lower cancellation rate of 27.76%.
- Resort Hotels have better booking retention.
- This indicates that hotel type is a strong determinant of booking cancellation behavior.

In [7]:
## OVERALL CANCELLATION RATE

duckdb.sql("""
SELECT 
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS overall_cancellation_rate
FROM hotel_data
""").df()


,total_bookings,cancelled_bookings,overall_cancellation_rate
0,119390,44224.0,37.04


### Observation
- Out of 119,390 bookings, 44,224 were cancelled.
- The overall cancellation rate is 37.04%.
- More than one-third of all reservations are cancelled before check-in.
- Such a high cancellation rate can significantly affect revenue forecasting and room inventory planning.
- Cancellation management can significantly improve revenue stability.
- Building a cancellation prediction model is justified.

In [8]:
## ADR(Average Daily Rate) BY HOTEL TYPE

duckdb.sql("""
SELECT
    hotel,
    ROUND(AVG(adr), 2) AS avg_adr,
FROM hotel_data
GROUP BY hotel
ORDER BY avg_adr DESC
""").df()


,hotel,avg_adr
0,City Hotel,105.30
1,Resort Hotel,94.95


### Observation
- City Hotels achieve an average ADR of 105.30.
- Resort Hotels achieve an average ADR of 94.95.
- City Hotels generate approximately 10.9% higher revenue per occupied room.
- Despite higher ADR, City Hotels also suffer higher cancellations.
- Revenue optimization and cancellation management should be jointly analyzed.

In [9]:
## CANCELLATION BY MARKET SEGMENT

duckdb.sql("""
SELECT 
    market_segment,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY market_segment
ORDER BY cancellation_rate DESC
""").df()


,market_segment,total_bookings,cancelled_bookings,cancellation_rate
0,Undefined,2,2.0,100.00
1,Groups,19811,12097.0,61.06
2,Online TA,56477,20739.0,36.72
3,Offline TA/TO,24219,8311.0,34.32
4,Aviation,237,52.0,21.94
5,Corporate,5295,992.0,18.73
6,Direct,12606,1934.0,15.34
7,Complementary,743,97.0,13.06


### Observation

- Group bookings exhibit the highest cancellation rate at 61.06%. 
- Online Travel Agency bookings contribute a cancellation rate of 36.72%.
- Direct bookings are considerably more reliable, with only 15.34% cancellations.
- Encouraging direct bookings could potentially reduce cancellation risk and improve revenue stability.

In [10]:
## CUSTOMER TYPE ANALYSIS

duckdb.sql("""
SELECT
    customer_type,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY customer_type
ORDER BY cancellation_rate DESC
""").df()


,customer_type,total_bookings,cancelled_bookings,cancellation_rate
0,Transient,89613,36514.0,40.75
1,Contract,4076,1262.0,30.96
2,Transient-Party,25124,6389.0,25.43
3,Group,577,59.0,10.23


### Observation

- Transient customers account for 89,613 bookings and exhibit the highest cancellation rate at 40.75%.
- Contract customers show a moderate cancellation rate of 30.96%.
- Group customers are the most reliable segment, with only 10.23% cancellations.
- Individual travelers appear substantially more likely to cancel than organized groups.

In [11]:
## DEPOSIT TYPE ANALYSIS

duckdb.sql("""
SELECT
    deposit_type,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled)*100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY deposit_type
ORDER BY cancellation_rate DESC
""").df()


,deposit_type,total_bookings,cancelled_bookings,cancellation_rate
0,Non Refund,14587,14494.0,99.36
1,No Deposit,104641,29694.0,28.38
2,Refundable,162,36.0,22.22


### Observation

- Non Refund bookings exhibit an extremely high cancellation rate of 99.36%.
- Out of 14,587 Non Refund bookings, 14,494 were cancelled.
- No Deposit bookings show a cancellation rate of 28.38%.
- Refundable bookings show the lowest cancellation rate at 22.22%.
- Deposit policy demonstrates one of the strongest relationships with cancellation behavior in the dataset.

In [12]:
## AVERAGE LEAD TIME BY CANCELLATION

duckdb.sql("""
SELECT
    is_canceled,
    ROUND(AVG(lead_time), 2) AS avg_lead_time
FROM hotel_data
GROUP BY is_canceled
""").df()

,is_canceled,avg_lead_time
0,1,144.85
1,0,79.98


### Observation
- Cancelled bookings are made approximately 145 days before arrival on average.
- Non-cancelled bookings are made approximately 80 days before arrival.
- Cancelled reservations have an average lead time that is nearly 81% higher.
- Customers booking far in advance are considerably more likely to cancel.

In [13]:
## STAY TYPE DISTRIBUTION

duckdb.sql("""
SELECT 
    stay_type,
    COUNT(*) AS bookings
FROM hotel_data
GROUP BY stay_type
ORDER BY bookings DESC
""").df()

,stay_type,bookings
0,Short Stay,76454
1,Medium Stay,37679
2,Long Stay,5257


### Observation
- Short Stay reservations account for 76,454 bookings, representing approximately 64% of all reservations.
- Medium Stay reservations account for 37,679 bookings.
- Long Stay reservations account for only 5,257 bookings.
- Hotel demand is overwhelmingly dominated by short-duration trips.

In [14]:
## STAY TYPE CANCELLATION RATE

duckdb.sql("""
SELECT 
    stay_type,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) *100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY stay_type
ORDER BY cancellation_rate DESC
""").df()

,stay_type,total_bookings,cancelled_bookings,cancellation_rate
0,Short Stay,76454,28820.0,37.70
1,Medium Stay,37679,13525.0,35.90
2,Long Stay,5257,1879.0,35.74


### Observation
- Short Stay bookings show a cancellation rate of 37.70%.
- Medium Stay bookings show a cancellation rate of 35.90%.
- Long Stay bookings show a cancellation rate of 35.74%.
- The variation across stay categories is less than 2 percentage points.
- Stay duration alone does not appear to be a major driver of cancellation behaviour.

In [15]:
## MONTHLY BOOKING TREND

duckdb.sql("""
SELECT 
    arrival_date_month,
    COUNT(*) AS bookings
FROM hotel_data
GROUP BY arrival_date_month
ORDER BY bookings DESC
""").df()

,arrival_date_month,bookings
0,August,13877
1,July,12661
2,May,11791
3,October,11160
4,April,11089
5,June,10939
6,September,10508
7,March,9794
8,February,8068
9,November,6794


### Observation
- August records the highest booking volume with 13,877 reservations.
- July follows closely with 12,661 bookings.
- January records the lowest demand with only 5,929 bookings.
- Peak demand occurs during the summer season (July–August), indicating strong seasonal travel patterns.
- August receives approximately 134% more bookings than January.
- Hotels can leverage these seasonal trends for dynamic pricing, staffing, and inventory planning.

In [17]:
## AVERAGE GUESTS PER BOOKING

duckdb.sql("""
SELECT
    ROUND(AVG(total_guests),2) AS avg_guests
FROM hotel_data
""").df()

,avg_guests
0,1.97


### Observation

- The average booking contains approximately 1.97 guests.
- Most reservations involve either one or two travelers.
- The dataset suggests that couples and small travel parties dominate hotel demand.
- Large group bookings represent only a small proportion of total reservations.
- Room allocation and marketing strategies should primarily target small occupancy groups.

In [18]:
## AVERAGE ADR BY CUSTOMER TYPE

duckdb.sql("""
SELECT 
    customer_type,
    ROUND(AVG(adr), 2) AS avg_adr
FROM hotel_data
GROUP BY customer_type
ORDER BY avg_adr DESC
""").df()

,customer_type,avg_adr
0,Transient,107.01
1,Contract,87.55
2,Transient-Party,86.08
3,Group,83.49


### Observation

- Transient customers generate the highest average daily revenue at 107.01 ADR.
- Group customers generate the lowest ADR at 83.49.
- Transient bookings produce approximately 28% higher room revenue than Group bookings.
- Although Transient customers contribute the highest revenue per booking, they also exhibit the highest cancellation rate (40.75%).
- This highlights a trade-off between revenue generation and booking reliability.

In [19]:
## AVERAGE TOTAL NIGHTS BY HOTEL

duckdb.sql("""
SELECT
    hotel,
    ROUND(AVG(total_nights),2) AS avg_nights
FROM hotel_data
GROUP BY hotel
""").df()

,hotel,avg_nights
0,Resort Hotel,4.32
1,City Hotel,2.98


### Observation

- Resort Hotel guests stay an average of 4.32 nights.
- City Hotel guests stay an average of 2.98 nights.
- Resort Hotel stays are approximately 45% longer than City Hotel stays.
- Resort Hotels appear to be associated with leisure and vacation travel, whereas City Hotels primarily serve shorter business or transit-related stays.
- Longer stays may contribute to more stable occupancy levels in Resort Hotels.

In [20]:
## TOP 10 COUNTRIES BY BOOKINGS

duckdb.sql("""
SELECT 
    country,
    COUNT(*) AS bookings,
FROM hotel_data
GROUP BY country
ORDER BY bookings DESC 
LIMIT 10
""").df()

,country,bookings
0,PRT,48590
1,GBR,12129
2,FRA,10415
3,ESP,8568
4,DEU,7287
5,ITA,3766
6,IRL,3375
7,BEL,2342
8,BRA,2224
9,NLD,2104


### Observation

- Portugal (PRT) is the largest customer market with 48,590 bookings.
- Portuguese customers account for approximately 40.7% of all reservations.
- The United Kingdom (GBR) and France (FRA) are the next largest markets with 12,129 and 10,415 bookings respectively.
- The hotel business relies heavily on domestic demand from Portugal.
- Geographic diversification opportunities may exist by targeting underrepresented international markets.

# Day 4 SQL Analytics Summary

## Key Business Insights

### 1. Cancellation Risk

- Overall cancellation rate is 37.04%.
- City Hotels experience a higher cancellation rate (41.73%) than Resort Hotels (27.76%).
- Long lead times are strongly associated with cancellations.

### 2. Customer Behaviour

- Transient customers have the highest cancellation rate (40.75%).
- Group customers are the most reliable segment with only 10.23% cancellations.
- Average booking contains approximately 2 guests.

### 3. Revenue Insights

- City Hotels generate higher ADR (105.30) than Resort Hotels (94.95).
- Transient customers produce the highest ADR (107.01).
- Higher revenue segments also tend to exhibit higher cancellation risk.

### 4. Booking Sources

- Group bookings show the highest cancellation rate (61.06%).
- Online Travel Agencies contribute significantly to cancellations.
- Direct bookings are substantially more reliable.

### 5. Stay Behaviour

- Short stays account for approximately 64% of all bookings.
- Stay duration has only a minor effect on cancellation rates.
- Resort Hotel guests stay significantly longer than City Hotel guests.

### 6. Geographic Demand

- Portugal is the dominant customer market, contributing over 40% of total bookings.
- The UK and France are the largest international markets.

## Conclusion

Hotel cancellations are primarily influenced by lead time, hotel type, deposit type, market segment, and customer type. These variables should be prioritized during predictive model development.